### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.
### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [11]:
from IPython.terminal.shortcuts.auto_suggest import Provider
import os 
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("gemini-3.1-flash-lite", model_provider="google_genai")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-google-genai': '4.5.2'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x0000020294DE85F0>, default_metadata=(), model_kwar

In [12]:
from pydantic import BaseModel,Field 
# BaseModel: Used to create a data model with defined fields and data types.
# Field: Used to add extra information, validation rules, and descriptions to fields.
class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie ")
    # title → Name of the field.
    # str → The field should contain a string.
    # Field() → Provides additional information about the field.
    #  description → Explains what the field represents.


#     How to use it
# movie = Movie(title="Interstellar")print(movie.title)


# Output:
# Interstellar

# If you provide an integer instead:
# movie = Movie(title=123)


# Pydantic will generally raise a validation error because title expects a string.
    year:int=Field(description="This is the release date of the movie")
    director:str=Field(description="This is the director of the movie")
    rating:float=Field(description="This is the movies rsting out of 10")
    

In [13]:
# 1. Invoke the raw model
response = model.invoke("Provide details of the movie Zero")

# 2. Get the unstructured text response
print(response.content)

[{'type': 'text', 'text': '*Zero* is a 2018 Indian Hindi-language romantic drama film directed by **Aanand L. Rai** and written by Himanshu Sharma. It is notable for being one of the most expensive Bollywood films ever produced at the time of its release and for its extensive use of visual effects.\n\nHere are the key details about the movie:\n\n### 1. Plot Summary\nThe film follows the life of **Bauua Singh**, a wealthy but vertically challenged man (a little person) living in Meerut. Bauua is searching for a companion and eventually encounters two women:\n*   **Aafia Bhinder:** A brilliant NASA scientist with cerebral palsy.\n*   **Babita Kumari:** A famous and struggling Bollywood actress.\n\nThe story explores Bauua’s journey as he navigates these two relationships, eventually leading to an unexpected mission involving space travel and Mars.\n\n### 2. Main Cast\n*   **Shah Rukh Khan** as Bauua Singh.\n*   **Anushka Sharma** as Aafia Bhinder.\n*   **Katrina Kaif** as Babita Kumari.\

In [17]:
structured_model = model.with_structured_output(Movie)
response = structured_model.invoke("Provide details of the movie Zero")
response

Movie(title='Zero', year=2018, director='Aanand L. Rai', rating=5.4)

### Messaged Output alongside parsed structure


In [19]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    name:str=Field("What is the name of the movie")
    date:int=Field("What is the release date")
    rating:float=Field("What is the rating of the movie")
model_with_structure=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure.invoke("Provide details for the movie Interstellar")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "Interstellar",\n  "date": 2014,\n  "rating": 8.7\n}', 'extras': {'signature': 'EqEBCp4BAWkUfRNEsrdRSocVoZxhT/Dbc3EjcQRU6r4UfEM5QzDP4toIyf4wbCtC91eFfwTEYBgLkuwHx3FFkD4ZH2bmfearJv2wEt078ddgI7I4HU/X8p3GMOapQG/anFWssekosc+u9UGJprhECHvkDZp4vybLnOtRd8DourgiucTy+G6fFrr5cndfAoFaVcQN7O5BoibZJoZBhuRpgLUgWpA='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a20369-6b58-76f6-a31e-e9e19512f46e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 32, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(name='Interstellar', date=2014, rating=8.7),
 'parsing_error': None}